# TokenMM Trade Data

Load local SQLite telemetry for fills, order actions, and quote cycles.

In [ ]:
from pathlib import Path
import os
import sqlite3

import pandas as pd

TELEMETRY_DIR = Path(
    os.environ.get("TOKENMM_TELEMETRY_DIR", "/var/lib/nautilus/telemetry/tokenmm")
)
DB_PATHS = {
    "execution_fill": TELEMETRY_DIR / "fills.sqlite",
    "order_action": TELEMETRY_DIR / "orders.sqlite",
    "quote_cycle": TELEMETRY_DIR / "quote_cycles.sqlite",
}
DB_PATHS


In [ ]:
def read_sqlite_table(path: Path, table: str, limit: int | None = 1000) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    query = f"SELECT * FROM {table}"
    if limit is not None:
        query += f" ORDER BY rowid DESC LIMIT {int(limit)}"
    with sqlite3.connect(path) as conn:
        return pd.read_sql_query(query, conn)


In [ ]:
fills = read_sqlite_table(DB_PATHS["execution_fill"], "execution_fill")
orders = read_sqlite_table(DB_PATHS["order_action"], "order_action")
quote_cycles = read_sqlite_table(DB_PATHS["quote_cycle"], "quote_cycle")

fills.head(), orders.head(), quote_cycles.head()


In [ ]:
order_columns = [
    "client_order_id",
    "quote_cycle_id",
    "reason_code",
    "decision_context_json",
]
fills_with_actions = fills.merge(
    orders[order_columns].drop_duplicates(),
    on=["client_order_id", "quote_cycle_id"],
    how="left",
    suffixes=("", "_order_action"),
)
fills_with_actions.head()


## Optional Postgres Swap

If the telemetry shipper is enabled, replace the SQLite helper with `pd.read_sql(...)` against the shipped `execution_fill`, `order_action`, and `quote_cycle` tables.